# Synapse serverless

La Function **orquesta** y Synapse **procesa**: la tarea envía SQL y recibe resultados pequeños (conteos,
métricas). Por qué y cuánto cuesta: [MOTOR_DE_COMPUTO.md](MOTOR_DE_COMPUTO.md).

In [ ]:
import logging
import os
import sys
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Raíz del repo (el notebook vive en docs/) y variables locales: .env y .env.synapse (ignorados por git).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))
for env_file in ('.env', '.env.synapse'):
    load_dotenv(ROOT / env_file, override=True)

# Solo en el notebook: ejecutar tareas que no están en el YAML. En la Function las tareas viven en el YAML.
os.environ['TAREA_DEBE_CONFIGURACION'] = 'false'

from centraal_dataframework.runner import reset_runner  # noqa: E402
from centraal_dataframework.settings import clear_settings_cache  # noqa: E402

clear_settings_cache()  # relee las variables recién cargadas
reset_runner()
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

HAY_STORAGE = bool(os.environ.get('AZURE_STORAGE_CONNECTION_STRING') or os.environ.get('STORAGE_ACCOUNT_NAME'))

# Requisitos (una vez por máquina):
#   pip install -e ".[synapse]"                     -> pyodbc
#   ODBC Driver 18 for SQL Server                   -> ver enlace abajo
#   az login [--tenant <tenant-id>]                 -> identidad, sin claves
#   su IP en el firewall del workspace              -> az synapse workspace firewall-rule create ...
#   usuario en cada base (lo ejecuta un admin; en la Function, el nombre de la identidad administrada):
#     CREATE USER [usuario@dominio.com] FROM EXTERNAL PROVIDER;
#     ALTER ROLE db_datareader ADD MEMBER [usuario@dominio.com];
#     GRANT VIEW DEFINITION TO [usuario@dominio.com];  -- validate_table lee la definición de la vista
#     -- si las vistas leen el lago con credencial: GRANT REFERENCES ON DATABASE SCOPED CREDENTIAL::[c] TO [...]
# Variables: copiar docs/synapse.env.example a .env.synapse en la raíz y completar.
# Driver: https://learn.microsoft.com/sql/connect/odbc/download-odbc-driver-for-sql-server
# Las secciones cuya variable SYNAPSE_TEST_* está vacía se saltan.
if not (os.environ.get('SYNAPSE_WORKSPACE') or os.environ.get('SYNAPSE_SERVER')):
    raise RuntimeError('Defina SYNAPSE_WORKSPACE en .env.synapse')

TABLA = os.environ.get('SYNAPSE_TEST_TABLE', '').strip()  # esquema.tabla para leer
DDL_SCHEMA = os.environ.get('SYNAPSE_TEST_DDL_SCHEMA', '').strip()  # esquema donde crear y borrar una vista
OTRA_BASE = os.environ.get('SYNAPSE_TEST_OTHER_DB', '').strip()  # otra base del workspace (p.ej. silver)
GX_QUERY = os.environ.get('SYNAPSE_TEST_GX_QUERY', '').strip()  # SELECT con columnas a validar y filtro de partición
GX_COLUMN = os.environ.get('SYNAPSE_TEST_GX_COLUMN', '').strip()  # una columna de GX_QUERY
TABLA_PARTICION = os.environ.get('SYNAPSE_TEST_PARTITION_TABLE', '').strip()  # vista particionada por carpeta
COL_PARTICION = os.environ.get('SYNAPSE_TEST_PARTITION_COLUMN', '').strip()  # su columna de partición
VAL_PARTICION = os.environ.get('SYNAPSE_TEST_PARTITION_VALUE', '').strip()  # una foto sana

import pyodbc  # noqa: E402

DRIVER = os.environ.get('SYNAPSE_ODBC_DRIVER') or 'ODBC Driver 18 for SQL Server'
assert DRIVER in pyodbc.drivers(), f'Instale {DRIVER} o ajuste SYNAPSE_ODBC_DRIVER'
print('tabla:', TABLA or '-', '| ddl:', DDL_SCHEMA or '-', '| otra base:', OTRA_BASE or '-', '| particionada:', TABLA_PARTICION or '-')

## 1. Cliente

In [ ]:
from centraal_dataframework.synapse import get_synapse, render

# Fuera de una tarea se usa get_synapse(); dentro, ctx.synapse.
# Solo con SYNAPSE_WORKSPACE el servidor es <workspace>-ondemand.sql.azuresynapse.net (serverless).
synapse = get_synapse()
print(synapse)  # sin password ni credencial

# EngineEdition 11 = serverless. Si falla con 18456 (login failed): falta el usuario en la base o el tenant es otro.
synapse.query(
    "SELECT SUSER_SNAME() AS usuario, DB_NAME() AS base, "
    "CAST(SERVERPROPERTY('EngineEdition') AS int) AS engine_edition"
)

## 2. Leer: pedir la respuesta, no los datos

In [ ]:
# scalar -> un valor | query -> lista de dicts | read_df -> DataFrame de pandas. Siempre resultados pequeños.
# Valores (fechas, ids): parámetros ? del driver. Identificadores (esquema, tabla): render con {{nombre}}.
print(synapse.query('SELECT ? AS texto, ? AS numero', 'a', 7))

# render solo acepta letras, números y _ separados por - : o . -> lo demás falla antes de llegar a Synapse.
try:
    render('SELECT * FROM {{tabla}}', tabla="x; DROP VIEW v")
except ValueError as exc:
    print('rechazado:', exc)

if TABLA:
    # Serverless cobra por datos procesados (mínimo 10 MB): COUNT o GROUP BY en SQL, no en pandas.
    print('filas:', synapse.scalar(render('SELECT COUNT_BIG(*) FROM {{tabla}}', tabla=TABLA)))
    print('columnas:', list(synapse.read_df(render('SELECT TOP 0 * FROM {{tabla}}', tabla=TABLA)).columns))

## 3. Tareas

In [ ]:
from centraal_dataframework import TaskContext, task


@task(uses=['synapse'])  # solo carga Synapse: no necesita storage
def contar_tabla(ctx: TaskContext):
    """Cuenta filas; devuelve un número, no los datos."""
    n = ctx.synapse.scalar(render('SELECT COUNT_BIG(*) FROM {{tabla}}', tabla=ctx.params['tabla']))
    ctx.logger.info('tabla=%s filas=%s', ctx.params['tabla'], n)  # métricas, nunca filas
    return n


# Agregar en SQL, por ejemplo:
#   ctx.synapse.query("SELECT canal, SUM(valor) AS total FROM gold.ventas WHERE fecha = ? GROUP BY canal", fecha)
if TABLA:
    print(contar_tabla(params={'tabla': TABLA}))

## 4. Transformar dentro de Synapse

In [ ]:
import tempfile

# Las transformaciones viven en archivos .sql versionados (carpeta sql/ del proyecto).
# execute_file: sustituye {{param}}, separa lotes por GO y ejecuta en autocommit (serverless no admite DDL en
# transacciones). Una vista no cuesta al crearse: se paga al consultarla.
SQL_DIR = Path(tempfile.mkdtemp())
(SQL_DIR / 'vista_muestra.sql').write_text(
    "CREATE OR ALTER VIEW {{esquema}}.cdf_ejemplo_{{run}} AS\n"
    "SELECT TOP 100 * FROM {{tabla}}\n"
    "GO\n",
    encoding='utf-8',
)

# Para materializar un resultado reutilizable, CETAS escribe Parquet en el lago (una carpeta por lote:
# DROP EXTERNAL TABLE no borra los archivos y CETAS falla si la carpeta existe):
#   CREATE EXTERNAL TABLE {{esquema}}.ventas_diarias_{{lote}}
#   WITH (LOCATION = 'gold/ventas_diarias/lote={{lote}}/', DATA_SOURCE = lago, FILE_FORMAT = parquet)
#   AS SELECT fecha, canal, SUM(valor) AS total FROM silver.ventas WHERE fecha = '{{lote}}' GROUP BY fecha, canal;


@task(uses=['synapse'])
def publicar_vista(ctx: TaskContext):
    """Crea o actualiza una vista."""
    p = ctx.params
    ctx.synapse.execute_file(SQL_DIR / 'vista_muestra.sql', esquema=p['esquema'], tabla=p['tabla'], run=p['run'])
    return f"{p['esquema']}.cdf_ejemplo_{p['run']}"


if TABLA and DDL_SCHEMA:
    vista = publicar_vista(params={'esquema': DDL_SCHEMA, 'tabla': TABLA, 'run': uuid.uuid4().hex[:8]})
    print(vista, 'filas:', synapse.scalar(f'SELECT COUNT_BIG(*) FROM {vista}'))
    synapse.execute(f'DROP VIEW IF EXISTS {vista}')  # limpieza

## 5. Varias bases y conexiones

In [ ]:
@task(uses=['synapse'])
def bases_de_la_tarea(ctx: TaskContext):
    """synapse_for: otra base del mismo workspace, misma credencial; ctx.synapse no cambia."""
    return ctx.synapse.scalar('SELECT DB_NAME()'), ctx.synapse_for(ctx.params['otra_base']).scalar('SELECT DB_NAME()')


if OTRA_BASE:
    print(bases_de_la_tarea(params={'otra_base': OTRA_BASE}))

# Otro workspace: conexión con nombre. uses=['synapse:gold'] -> ctx.resource('synapse', 'gold'), con variables
# SYNAPSE_WORKSPACE__GOLD, SYNAPSE_DATABASE__GOLD, ... (no hereda workspace, base ni credenciales).

## 6. Calidad: GX en Synapse (plan A) o en pandas (plan B)

`ctx.gx.validate_table` corre la suite **en el motor** sobre una tabla o vista con nombre: GX traduce cada expectativa a SQL y solo vuelven agregados. Con `modo='pandas'` la misma suite valida la tabla traída con `read_df` (las filas viajan a la Function). El motor nunca cambia solo.

- En bronze se valida **la foto de un día**: `particion={'fecha': valor}`. La librería comprueba contra la definición de la vista que la columna sale de `filepath()` (hace falta `GRANT VIEW DEFINITION`) y aplica `WHERE fecha = ?` sin funciones, sin sondear la tabla entera ni listar valores distintos: serverless lee solo esa carpeta. Una vista particionada sin `particion` se rechaza; `particion='completa'` la valida entera a propósito.
- Los cruces (un SELECT) van por `ctx.gx.validate_query(..., modo='pandas')`; en `sql` solo admite expectativas de tabla.
- `result_format='SUMMARY'` por defecto; `COMPLETE` vuelve a escanear.
- `medir_costo=True` deja `data_processed_mb` en el resultado (mínimo 10 MB por consulta: no sirve para medir la poda; para eso está la prueba con la carpeta dañada de `pytest -m synapse`).

Guía completa: [MOTOR_DE_COMPUTO.md](MOTOR_DE_COMPUTO.md).


In [ ]:
@task(uses=['synapse', 'gx'])
def dq_tabla(ctx: TaskContext):
    """La foto del día de una vista particionada; ctx.params['modo'] = 'pandas' cambia al plan B sin tocar la suite."""
    import great_expectations as gx

    columna = ctx.params['columna']
    expectativas = [
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=1),
        gx.expectations.ExpectColumnValuesToNotBeNull(column=columna, mostly=0.95),
    ]
    return ctx.gx.validate_table(
        ctx.synapse,
        name='tabla',
        tabla=ctx.params['tabla'],
        expectations=expectativas,
        particion={columna: ctx.params['valor']},
        modo=ctx.params.get('modo', 'sql'),
        medir_costo=True,
    )


@task(uses=['synapse', 'gx'])
def dq_cruce(ctx: TaskContext):
    """Un SELECT (cruce) se valida en pandas: las filas viajan a la Function."""
    import great_expectations as gx

    expectativas = [gx.expectations.ExpectColumnValuesToNotBeNull(column=ctx.params['columna'], mostly=0.95)]
    return ctx.gx.validate_query(ctx.synapse, name='cruce', query=ctx.params['query'], expectations=expectativas, modo='pandas')


# Más sobre GX, fallos y silencios: 02_great_expectations.ipynb
if TABLA_PARTICION and COL_PARTICION and VAL_PARTICION:
    for modo in ('sql', 'pandas'):
        r = dq_tabla(params={'tabla': TABLA_PARTICION, 'columna': COL_PARTICION, 'valor': VAL_PARTICION, 'modo': modo})
        print(f'{modo}: success={r.success} failed={r.failed} MB procesados={r.data_processed_mb}')
else:
    print('SALTADO: configure SYNAPSE_TEST_PARTITION_* en .env.synapse')
if GX_QUERY and GX_COLUMN:
    r = dq_cruce(params={'query': GX_QUERY, 'columna': GX_COLUMN})
    print(f'cruce pandas: success={r.success} failed={r.failed}')


## 7. Runner

In [ ]:
from centraal_dataframework.runner import get_runner

# En la Function las tareas se programan en el YAML:
#   tareas:
#     publicar_ventas_diarias: {horas: '6', params: {esquema: gold}}
#     dq_ventas_diarias: {depends_on: [publicar_ventas_diarias], params: {tabla: gold.ventas_diarias}}
# App settings: SYNAPSE_WORKSPACE, SYNAPSE_DATABASE, SYNAPSE_AUTH=identity.
runner = get_runner()
print(runner.list_tasks())
print(runner.check_resources() or 'recursos configurados')
if TABLA:
    print(runner.run_task('contar_tabla', params={'tabla': TABLA}, notify=False))

## 8. pandas y polars: solo con resultados pequeños

In [ ]:
# TOP en el SQL, no .head() después: si no, igual se lee y transfiere todo.
if TABLA:
    display(synapse.read_df(render('SELECT TOP 10 * FROM {{tabla}}', tabla=TABLA)))  # noqa: F821

    # polars no es dependencia de la librería: se construye desde query.
    try:
        import polars as pl

        print(pl.DataFrame(synapse.query(render('SELECT TOP 10 * FROM {{tabla}}', tabla=TABLA))))
    except ImportError:
        print('polars no está instalado (opcional)')